# Generate the chronological factory datasetChronological generation workflow for the Sprint 11 factory simulation: a deterministic 3-6 month calendar of routed operation events over shared robots, with robot health, failures, maintenance, precursor quarantine, and chronological splits.Key features:- Direct parameters: configure generation in the notebook via `ChronicleParams` (e.g. `params = ChronicleParams(profile="server", seed=0)`) or the `V2_CHRONICLE_PROFILE` / `V2_SEED` / `V2_CHRONICLE_ROOT` environment variables.- Manual paths: run from the repository root and set `SRC_DIR` and `V2_CHRONICLE_ROOT` in the first cells; the exact configured paths are used with fail-fast errors naming the variable to change.- Public entry point only: `client_config` / `server_config` plus `materialize_chronological` (equivalent CLI printed for server runs); no scheduler, health, signal, or split internals are reimplemented here.- Schedule/split/quarantine inspection: per-robot non-overlap, asynchronous cross-robot timelines, verified-healthy non-quarantined development views, and static/temporal test membership from the persisted manifest.- Bounded diagnostics: counts, per-robot/program prevalence, shard checksums, a 4-file decode probe, and one small `chronicle-summary.json`; bulk bytes stay in shards.- Determinism: `PYTHONHASHSEED=0` plus one explicit seed drives every stream; the manifest records per-stream seeds and the config hash.

In [ ]:
import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

os.environ.setdefault("PYTHONHASHSEED", "0")

# ---- Repository source (edit these one-line values; used exactly) ----
# Run notebooks from the repository root, or set V2_REPO_ROOT to the checkout path.
V2_REPO_ROOT = os.environ.get("V2_REPO_ROOT", ".")
SRC_DIR = os.environ.get("V2_SRC_DIR", str(Path(V2_REPO_ROOT) / "src"))

_source_dir = Path(SRC_DIR).expanduser()
if not (_source_dir / "synth").is_dir():
    raise FileNotFoundError(
        f"Repository source not found: SRC_DIR={_source_dir} has no 'synth' package. "
        "Run from the repository root or set V2_REPO_ROOT / V2_SRC_DIR to the checkout.")
_source_resolved = str(_source_dir.resolve())
if _source_resolved not in sys.path:
    sys.path.insert(0, _source_resolved)
print(f"[Env] Loaded synth modules from: {_source_dir}")

from synth.chronicle import client_config, load_chronological, materialize_chronological, server_config

In [ ]:
# ---- Explicit roots / profile / seed (edit these one-line values; used exactly) ----
CHRONICLE_ROOT = os.environ.get("V2_CHRONICLE_ROOT", "data/generated/chronicle-client")
CHRONICLE_PROFILE = os.environ.get("V2_CHRONICLE_PROFILE", "client")
CHRONICLE_SEED = int(os.environ.get("V2_SEED", "0"))
SHARD_SIZE = int(os.environ.get("V2_SHARD_SIZE", "64"))
OVERWRITE = os.environ.get("V2_OVERWRITE", "false").lower() in ("true", "1", "yes")

@dataclass
class ChronicleParams:
    """Chronological generation configuration (works from a repository checkout)."""
    root: str = CHRONICLE_ROOT
    profile: str = CHRONICLE_PROFILE
    seed: int = CHRONICLE_SEED
    shard_size: int = SHARD_SIZE
    overwrite: bool = OVERWRITE

params = ChronicleParams()
if params.profile not in ("client", "server"):
    raise ValueError(f"Unknown chronological profile {params.profile!r}; expected 'client' or 'server'.")
if not params.root.strip():
    raise ValueError("Chronological root must be an explicit path; set V2_CHRONICLE_ROOT.")
if params.shard_size <= 0:
    raise ValueError("SHARD_SIZE must be positive.")

DATA_ROOT = Path(params.root).expanduser()
DATA_ROOT = DATA_ROOT if DATA_ROOT.is_absolute() else Path.cwd() / DATA_ROOT
MANIFEST_PATH = DATA_ROOT / "manifest.json"
if MANIFEST_PATH.exists() and not params.overwrite:
    raise FileExistsError(
        f"Manifest already exists at {MANIFEST_PATH}; set V2_OVERWRITE=true to replace it.")

cfg = client_config(seed=params.seed) if params.profile == "client" else server_config(seed=params.seed)
print(f"[Config] profile={params.profile} seed={params.seed} root={DATA_ROOT} shard_size={params.shard_size}")
print("Equivalent CLI: uv run python -m synth.cli --chronological "
      f"--profile {params.profile} --output {DATA_ROOT} --seed {params.seed} --shard-size {params.shard_size}")

In [ ]:
# Generate the chronological dataset through the public entry point.
manifest = materialize_chronological(cfg, DATA_ROOT, shard_size=params.shard_size, overwrite=params.overwrite)
counts = manifest["counts"]
calendar = manifest["calendar"]
print(f"[Materialize] total={counts['total']} normal={counts['normal']} abnormal={counts['abnormal']} "
      f"dev_train={counts['dev_train']} dev_val={counts['dev_val']} "
      f"test_static={counts['test_static']} test_temporal={counts['test_temporal']} "
      f"quarantined={counts['quarantined']}")
print(f"[Materialize] cutoff_time={calendar['cutoff_time']} quarantine_s={calendar['quarantine_s']} "
      f"config_hash={manifest['config_hash']} generator={manifest['generator_version']}")
print(f"[Materialize] seeds={manifest['seeds']}")

In [ ]:
# Reload deterministically and audit schedule / split / quarantine invariants.
from collections import defaultdict

samples, reloaded = load_chronological(DATA_ROOT)
if reloaded["config_hash"] != manifest["config_hash"]:
    raise RuntimeError("Manifest changed between materialization and reload; refusing to report.")
for key in ("format", "generator_version", "config_hash", "resolved_config", "seeds", "counts",
            "calendar", "schedule", "episodes", "splits", "files", "shards"):
    if key not in reloaded:
        raise RuntimeError(f"Chronicle manifest at {MANIFEST_PATH} is missing required key {key!r}.")
splits = reloaded["splits"]
for view in ("dev_train", "dev_val", "test_static", "test_temporal", "quarantined"):
    if view not in splits or not isinstance(splits[view], list):
        raise RuntimeError(f"Chronicle manifest splits are missing view {view!r}.")
    if view != "quarantined" and not splits[view]:
        raise RuntimeError(f"Chronicle split view {view!r} is empty; regenerate the {params.profile} profile.")
by_id = {row["file_id"]: row for row in reloaded["files"]}
for file_id in set(splits["dev_train"]) | set(splits["dev_val"]):
    row = by_id[file_id]
    if row["file_label"] != "normal" or row["is_quarantined"]:
        raise RuntimeError(f"Development views hold verified-healthy non-quarantined files only; got {row}.")
    if "dev-train" not in row["member_views"] and "dev-val" not in row["member_views"]:
        raise RuntimeError(f"Development file {file_id} carries no dev member view: {row['member_views']}.")
# One robot processes at most one operation at a time: per-robot intervals are disjoint.
per_robot = defaultdict(list)
for event in reloaded["schedule"]:
    per_robot[event["robot_id"]].append((event["start_time"], event["end_time"], event["operation_id"]))
for robot_id in sorted(per_robot):
    ordered = sorted(per_robot[robot_id])
    for (_, end_a, op_a), (start_b, _, op_b) in zip(ordered, ordered[1:]):
        if end_a > start_b:
            raise RuntimeError(f"Overlapping operations on {robot_id}: {op_a} and {op_b}.")
    print(f"[Schedule] {robot_id}: {len(ordered)} operations, no overlaps, span [{ordered[0][0]}, {ordered[-1][1]}]")
spans = [(min(s for s, _, _ in v), max(e for _, e, _ in v)) for v in per_robot.values()]
interleaved = any(a0 < b1 and b0 < a1 for a0, a1 in spans for b0, b1 in spans if (a0, a1) != (b0, b1))
print("[Schedule] cross-robot timelines interleave asynchronously." if interleaved
      else "[Schedule] note: robot timelines do not interleave; expected asynchronous execution.")
print(f"[Quarantine] {len(splits['quarantined'])} precursor-quarantined files excluded from development views.")

In [ ]:
# Bounded diagnostics: prevalence, shard integrity, a 4-file probe, and one small summary.
from collections import Counter

robot_counts = Counter(row["robot_id"] for row in reloaded["files"])
program_counts = Counter(row["program_id"] for row in reloaded["files"])
label_counts = Counter(row["file_label"] for row in reloaded["files"])
print(f"[Diagnostics] files={len(reloaded['files'])} shards={len(reloaded['shards'])} labels={dict(label_counts)}")
for robot_id in sorted(robot_counts):
    print(f"[Diagnostics] {robot_id}: {robot_counts[robot_id]} files")
for program_id in sorted(program_counts):
    print(f"[Diagnostics] {program_id}: {program_counts[program_id]} files")
for shard in reloaded["shards"]:
    for key in ("path", "start", "end", "count", "file_ids", "sha256"):
        if key not in shard:
            raise RuntimeError(f"Shard entry is missing required key {key!r}.")
print(f"[Diagnostics] shard0={reloaded['shards'][0]['path']} sha256={reloaded['shards'][0]['sha256'][:16]}...")
# Decode only a bounded probe (K=4); bulk bytes stay in shards.
probe = samples[:4]
if not probe:
    raise RuntimeError(f"Chronological dataset at {DATA_ROOT} decoded to zero files.")
for sample in probe:
    sample.validate()
    channels, length = sample.x.shape
    print(f"[Probe] {sample.file_id} C={channels} T={length} label={sample.file_label.value}")
OUTPUT_ROOT = Path(os.environ.get("V2_OUTPUT_ROOT", str(DATA_ROOT / "notebook-output")))
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
summary = {
    "profile": params.profile, "seed": params.seed, "data_root": str(DATA_ROOT),
    "manifest_path": str(MANIFEST_PATH), "config_hash": reloaded["config_hash"],
    "generator_version": reloaded["generator_version"], "counts": counts, "calendar": calendar,
    "per_robot_counts": dict(sorted(robot_counts.items())), "per_program_counts": dict(sorted(program_counts.items())),
    "probe_file_ids": [s.file_id for s in probe],
}
summary_path = OUTPUT_ROOT / "chronicle-summary.json"
summary_path.write_text(json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
print(f"[Output] wrote {summary_path} with schema {sorted(summary)}")